# 01 — Ingest + Clean (PRODUCTION PIPELINE)

**What this notebook is:** the actual reusable IBP-cycle pipeline. This is what gets deployed and scheduled in production, and its logic lives entirely in `src/` — this notebook only orchestrates it.

**Its only assumption about the input:** raw CSVs matching `schema.raw.*` already exist at `<data_root>/raw/`. It does not care whether they came from `generate_data.py` (notebook 00, this capstone) or a real ERP extract (production). That boundary is deliberate — nothing in `src/ingest.py` or `src/cleaner.py` references the generator.

**Prerequisite:** run `00_generate_test_data.ipynb` first in this same Colab session (or upload `data_primary/raw/` and `data_control/raw/` from a prior run).

**Output of this notebook — the pipeline's real deliverables:**
- `<data_root>/clean/clean_master.parquet` — the canonical clean table
- `<data_root>/clean/dq_report.csv` / `.md` — the data quality report
- `<data_root>/clean/Step4_Data_Quality_Review.xlsx` — the human sign-off artefact. **Step 5 does not begin until this workbook is reviewed and signed off.**

## Setup — confirm the repo and raw data are present

In [ ]:
import os, sys

REPO = '/content/ibp-tradeoff'
os.chdir(REPO)
sys.path.insert(0, REPO)

for tag in ['data_primary', 'data_control']:
    raw_dir = os.path.join(REPO, tag, 'raw')
    if not os.path.isdir(raw_dir):
        raise FileNotFoundError(
            f"{raw_dir} not found. Run 00_generate_test_data.ipynb first, "
            "in this same session, before this notebook."
        )
print('Raw data found for both datasets. Proceeding.')


## Ingest + clean both datasets
Uses `src/ingest.py` (`DataIngestor`), `src/cleaner.py` (`DataCleaner`), and `src/report_step4.py` (`build_review_workbook`). Same code, both datasets, zero edits — this is the reusable-pipeline evidence.

Expect on both: rows out **2,159** · skus **60** · nulls **87** · recovered **86** · roll-forward max **0.000007**

In [ ]:
import pandas as pd
from src.ingest import DataIngestor
from src.cleaner import DataCleaner
from src.report_step4 import build_review_workbook

results = {}
dq_rows = []

for data_root in ['data_primary', 'data_control']:
    print('=' * 78)
    print(f'DATASET: {data_root}')
    print('=' * 78)

    ingestor = DataIngestor(repo_root=REPO, data_root=data_root)
    raw = ingestor.load()

    cleaner = DataCleaner(ingestor.schema, ingestor.assumptions)
    df_clean, sku_master_clean, dq_report = cleaner.clean(raw)

    rows_in = sum(v for k, v in ingestor.rows_in_by_source.items() if k != 'sku_master')
    print('\nRECONCILIATION')
    for source, n in ingestor.rows_in_by_source.items():
        print(f'  rows in  . {source:<12} {n:>6,}')
    print(f'  rows in  . transactional total   {rows_in:>6,}')
    for d in cleaner.dropped:
        print(f"  dropped  . {d['reason']:<45} {d['rows']:>3}")
    print(f'  rows out . clean_master          {len(df_clean):>6,}')

    print('\nDATA QUALITY REPORT')
    print(dq_report.to_string(index=False))

    out_dir = os.path.join(REPO, data_root, 'clean')
    os.makedirs(out_dir, exist_ok=True)

    parquet_path = os.path.join(out_dir, 'clean_master.parquet')
    df_clean.to_parquet(parquet_path, index=False)

    dq_csv_path = os.path.join(out_dir, 'dq_report.csv')
    dq_md_path = os.path.join(out_dir, 'dq_report.md')
    dq_report.to_csv(dq_csv_path, index=False)
    with open(dq_md_path, 'w') as f:
        f.write(f'# Data Quality Report — {data_root}\n\n')
        f.write('Generated by `src/cleaner.py` (`DataCleaner`), Step 4.\n')
        f.write('Implements `cleaning-spec.md` C-01 to C-13.\n\n')
        f.write(dq_report.to_markdown(index=False))
        f.write('\n')

    xlsx_path = os.path.join(out_dir, 'Step4_Data_Quality_Review.xlsx')
    build_review_workbook(
        raw=raw, df_clean=df_clean, master=sku_master_clean, dq_report=dq_report,
        dropped=cleaner.dropped, out_path=xlsx_path, dataset_label=data_root,
    )

    print(f'\nwrote {parquet_path}')
    print(f'wrote {dq_csv_path}')
    print(f'wrote {dq_md_path}')
    print(f'wrote {xlsx_path}')

    dq_tagged = dq_report.copy()
    dq_tagged.insert(0, 'dataset', data_root)
    dq_rows.append(dq_tagged)
    results[data_root] = (df_clean, sku_master_clean, dq_report)

combined_dq = pd.concat(dq_rows, ignore_index=True)
combined_path = os.path.join(REPO, 'data_quality_summary.csv')
combined_dq.to_csv(combined_path, index=False)
print(f'\nwrote {combined_path}')

print('\n' + '=' * 78)
print('REUSABILITY CHECK — same DataCleaner, two datasets, no code change')
for tag, (df, _, dq) in results.items():
    d = dict(zip(dq.metric, dq.value))
    print(
        f"  {tag:<14} rows={int(d['rows_out']):,}  skus={int(d['skus_out'])}  "
        f"nulls={int(d['nulls_in_volume'])}  recovered={int(d['nulls_recovered_by_identity'])}  "
        f"rollforward_max={d['stock_rollforward_max_abs_diff']:.6f}"
    )

print()
print('>>> ACTION REQUIRED before Step 5:')
print('>>> Download data_primary/clean/Step4_Data_Quality_Review.xlsx, review it,')
print('>>> and complete the sign-off on sheet 1 (Approved / Approved with comments / Rejected).')

df_clean = results['data_primary'][0]


## (Optional) Run the regression + robustness test suite

In [ ]:
sh_r = __import__('subprocess').run('pip install pytest -q', shell=True, capture_output=True, text=True)
print(sh_r.stdout[-500:])
r = __import__('subprocess').run('python -m pytest tests/ -v', shell=True, cwd=REPO, capture_output=True, text=True)
print(r.stdout[-3000:])


## (Optional) Push the clean outputs and review workbook to GitHub
Uncomment once you've signed off the workbook. This commits the *approved* artefact, not a draft.

In [ ]:
# sh_git = __import__('subprocess').run
# import subprocess
# subprocess.run('git config --global user.email "your@email.com"', shell=True, cwd=REPO)
# subprocess.run('git config --global user.name "rdelolmog-creator"', shell=True, cwd=REPO)
# subprocess.run('git add data_primary/clean data_control/clean data_quality_summary.csv', shell=True, cwd=REPO)
# subprocess.run('git commit -m "Step 4 signed off: clean_master + review workbook, both datasets"', shell=True, cwd=REPO)
# subprocess.run('git push', shell=True, cwd=REPO)
